# Pin a claim by hash and watch an approval lapse

*Adversarial Mathematics From First Principles*, [chapter 20](https://programmer.ie/books/math/20-chapter/) · [worked example](https://programmer.ie/examples/math/20-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

Every investigation starts with a claim contract, `investigation.json`, which
fixes the statements, checks and controls before anything runs. Every run
records the sha256 of the contract's bytes, and every approval names the hash
it approves. This program builds a cut-down contract in the same shape and
shows what the hash does and does not protect.

## Run it

Standard library only; nothing to install.

In [1]:
"""The claim contract: statements pinned, and pinned by hash (standard library only).

A cut-down contract in the shape of the laboratory's investigation.json. The real
runner records sha256 of the contract file's bytes; so does this program.
"""
import hashlib
import json

contract = {
    "id": "DEMO-01",
    "contract_version": 1,
    "claim": "For every natural number n, 2^(2^n) + 1 is prime.",
    "statements": {"P01": "2 ^ (2 ^ 5) + 1 = 641 * 6700417", "P02": "¬ Nat.Prime (2 ^ (2 ^ 5) + 1)"},
    "checks": [{"id": "compute", "expect_stdout": "refuted"}],
    "controls": [{"id": "wrong-divisor", "change": "641 -> 643", "expect": "rejected"}],
    "correspondence": {"id": "DEMO-CORR-01", "status": "pending"},
}


def file_bytes(c):
    return (json.dumps(c, indent=2, ensure_ascii=False) + "\n").encode("utf-8")


def sha(c):
    return hashlib.sha256(file_bytes(c)).hexdigest()


h1 = sha(contract)
print(f"contract version 1, sha256 {h1}")

approval = {"reviewer_role": "user", "decision": "approve", "correspondence": "DEMO-CORR-01", "contract_sha256": h1}
print(f"an approval recorded against it applies: {approval['contract_sha256'] == h1}")

edited = json.loads(json.dumps(contract))
edited["statements"]["P02"] = "¬ Nat.Prime (2 ^ (2 ^ 4) + 1)"           # n = 4: 65537, which IS prime
h2 = sha(edited)
print(f"\nedit one statement: sha256 {h2}")
print(f"the old approval applies to the edited contract: {approval['contract_sha256'] == h2}  -> status falls back to pending")

spaced = file_bytes(contract).replace(b'"id": "DEMO-01"', b'"id":  "DEMO-01"')
print(f"\nadd one space: sha256 {hashlib.sha256(spaced).hexdigest()}")
print("a hash pins the bytes, not the meaning: any change at all is a new version.")

print("\nWhat a contract cannot do: the hash shows which contract a run used, not that it is right.")
print("The edited P02 says 65537 is not prime. It has no divisor from 2 to 256:",
      all(65537 % d for d in range(2, 257)), "-- so 65537 is prime, and P02 would be refuted.")

contract version 1, sha256 670278cc839f7701b4fca5e25c6257f7356a9277b5881fd2ada46e5fde61cb19
an approval recorded against it applies: True

edit one statement: sha256 854d085d80c3790e98696eed0e2aab21f643c3907ad790d27dbbc2d0aefba8f0
the old approval applies to the edited contract: False  -> status falls back to pending

add one space: sha256 f486235f7ec88a2dfa9792645df985e2b42458dedc812945544b6a0aae02399a
a hash pins the bytes, not the meaning: any change at all is a new version.

What a contract cannot do: the hash shows which contract a run used, not that it is right.
The edited P02 says 65537 is not prime. It has no divisor from 2 to 256: True -- so 65537 is prime, and P02 would be refuted.


## What the output shows

Editing one statement gives a new hash, so the author's approval no longer
applies and the status falls back to pending. Adding a single space does the
same: the hash pins bytes, not meaning. And a pinned contract can still say
something false: the edited statement claims 65537 is not prime, and 65537 is
prime.

## Not shown

- This is a cut-down contract in the shape of the laboratory's `investigation.json`, not one of its contracts.
- Hashing shows which contract a run used; it says nothing about whether the contract's statements are the right ones.

## Challenge it

Examine why the laboratory records the contract's hash in every run and in
every approval. What changes, and what does not, when a statement is edited, a
space is added, or the file is re-saved with different line endings? Propose a
contract edit that should not invalidate an approval, and say whether the
laboratory's rule handles it.